# LLMs in Production — AI Lab Instructor Notebook

*ML System Design · Expert*



4 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import json
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

def pretty(d):
    print(json.dumps(d, indent=2, sort_keys=True))

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# LLMs in Production — Student Lab

Offline lab: study two common production optimization ideas using a toy classifier that stands in for an LLM head.

In this lab you will:
- build a synthetic teacher model
- quantize the teacher weights with fake int8
- distill the teacher into a smaller student model
- compare quality, memory, and tradeoffs

Run the tasks in order. Later sections depend on the teacher logits created earlier.

In [ ]:
pretty({'lab': 'llms in production', 'focus': ['latency / memory tradeoffs', 'quantization', 'distillation', 'quality vs cost']})

**Why this works.** This lab is about production tradeoffs, not model invention. The main question is how to make a strong model cheaper or faster without losing too much quality.

## Task 2: 0 — Synthetic dataset + teacher model

## 0 — Synthetic dataset + teacher model

Create a sparse synthetic dataset and a stronger teacher model. The teacher logits act as the high-quality baseline that later sections will compress.

In [ ]:
$f

**Why this works.** The teacher is the expensive reference model. Everything later is measured relative to it, so the first step is to create a strong baseline and keep its logits available.

## Task 3: 1 — Quantization (fake int8)

## 1 — Quantization (fake int8)

Quantize the teacher weights to int8, dequantize them back for inference, and compare memory versus quality. This is a toy stand-in for production post-training quantization.

In [ ]:
def quantize_int8(W: np.ndarray):
    max_abs = float(np.max(np.abs(W)))
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    W_q = np.clip(np.round(W / scale), -127, 127).astype(np.int8)
    return W_q, float(scale)


def dequantize_int8(W_q: np.ndarray, scale: float):
    return W_q.astype(np.float64) * float(scale)


def model_bytes_float64(W: np.ndarray, b: np.ndarray) -> int:
    return int(W.size * 8 + b.size * 8)


def model_bytes_int8(W_q: np.ndarray, b: np.ndarray) -> int:
    return int(W_q.size * 1 + b.size * 8)

Wq, scale = quantize_int8(Wt)
Wt_hat = dequantize_int8(Wq, scale)
logits_q = X @ Wt_hat + bt
acc_teacher = accuracy_from_logits(teacher_logits, y)
acc_q = accuracy_from_logits(logits_q, y)

print('acc_teacher', acc_teacher)
print('acc_q', acc_q)
print('bytes_fp64', model_bytes_float64(Wt, bt))
print('bytes_int8', model_bytes_int8(Wq, bt))
check('size_reduced', model_bytes_int8(Wq, bt) < model_bytes_float64(Wt, bt))
check('quant_acc_close', acc_teacher - acc_q < 0.02)

**Why this works.** Quantization keeps the model architecture the same but stores weights in a cheaper format. In production, the appeal is usually lower memory footprint and better throughput with only a small quality drop.

## Task 4: 2 — Distillation (smaller student matches teacher logits)

## 2 — Distillation (smaller student matches teacher logits)

Fit a smaller student model to the teacher logits. In this toy lab, the student only uses the first `d_student` input dimensions, which stands in for a smaller hidden width or a narrower network.

In [ ]:
$10

**Why this works.** Distillation changes the model itself, not just the number format. A smaller student learns to imitate the teacher, and the engineering question becomes whether the quality loss is worth the savings.